# Problema de las 8 Reinas - Algoritmo Genético

A continuación, se definen y explican las 4 partes fundamentales del algoritmo genético implementado en el archivo `TP4_6.py` para resolver el problema de las 8 reinas.

---

### a. Codificación de los individuos
**Teoría:** En el problema de las 8 Reinas, un individuo (un posible estado del tablero) se codifica como un arreglo o lista de 8 números enteros. 
- Cada **índice** de la lista (de 0 a 7) representa una **columna** del tablero.
- El **valor** en ese índice representa la **fila** (de 0 a 7) en la que se encuentra la reina para esa columna.

Esta codificación es muy eficiente porque asegura implícitamente que nunca habrá dos reinas en la misma columna, reduciendo el espacio de búsqueda significativamente.

In [ ]:
import numpy as np

# Generación de un individuo aleatorio (como en la población inicial del AGS)
# Se genera una lista de 8 enteros, donde cada valor está entre 0 y 7
individuo = np.random.randint(0, 8, 8).tolist()
print("Ejemplo de individuo codificado:", individuo)

--- 
### b. La función de selección natural
**Teoría:** La selección natural determina qué individuos son aptos para reproducirse. En `TP4_6.py`, se utiliza una **Selección por Torneo**. 
Consiste en tomar una muestra aleatoria pequeña de la población (en el código, 3 individuos) y elegir al que tenga mejor "fitness". En este problema, el fitness se evalúa con la función `calcular_ataques`: a menor cantidad de ataques (pares de reinas amenazándose), mejor fitness.
Además, el algoritmo usa **Elitismo**, ya que el mejor individuo de cada generación siempre pasa intacto a la siguiente (`new_pop = [best_estado.copy()]`).

In [ ]:
import random

def calcular_ataques(estado):
    """
    Calcula el número de pares de reinas que se amenazan.
    A menor cantidad de ataques, mejor fitness.
    """
    ataques = 0
    n = len(estado)
    for i in range(n):
        for j in range(i + 1, n):
            if estado[i] == estado[j] or abs(estado[i] - estado[j]) == abs(i - j):
                ataques += 1
    return ataques

def seleccion_torneo(pop, ataques_pop, k=3):
    """
    Selecciona un padre mediante torneo de tamaño k.
    Basado en las líneas 70-74 de TP4_6.py
    """
    torneo_indices = random.sample(range(len(pop)), k)
    # El ganador es el índice que tenga el menor número de ataques
    ganador_idx = min(torneo_indices, key=lambda x: ataques_pop[x])
    return pop[ganador_idx]

--- 
### c. La función de combinación
**Teoría:** La función de combinación (o cruzamiento/crossover) toma el material genético de dos padres y lo une para crear un descendiente (hijo). 
En el código se implementa un **Cruzamiento de un Punto** (Single-point Crossover). Se elige un punto de corte aleatorio en la lista del individuo (entre 1 y 6). El hijo hereda todos los genes anteriores al punto de corte del primer padre, y todos los genes desde el punto de corte en adelante del segundo padre.

In [ ]:
def combinacion_un_punto(p1, p2):
    """
    Combina dos padres eligiendo un punto de corte aleatorio.
    Basado en las líneas 76-77 de TP4_6.py
    """
    # Se elige punto de corte entre el índice 1 y el 6 (para listas de largo 8)
    pt = random.randint(1, 6)
    
    # El hijo concatena la parte inicial de p1 y la parte final de p2
    hijo = p1[:pt] + p2[pt:]
    return hijo

---
### d. La función de mutación
**Teoría:** La mutación añade alteraciones aleatorias a los hijos generados, lo cual es vital para mantener la diversidad genética y evitar que el algoritmo se estanque en mínimos locales.
En este caso, cada hijo generado tiene una cierta probabilidad de mutar (tasa de mutación, e.g., 20%). Si se activa la mutación, se selecciona una columna al azar y se cambia la posición de la reina a una nueva fila aleatoria.

In [ ]:
def mutacion(hijo, mut_rate=0.2):
    """
    Aplica mutación a un individuo con cierta probabilidad.
    Basado en las líneas 79-82 de TP4_6.py
    """
    # Si el número aleatorio cae dentro del ratio de mutación (ej. < 0.2)
    if random.random() < mut_rate:
        col_mutar = random.randint(0, 7)    # Elegir columna al azar
        fila_nueva = random.randint(0, 7)   # Elegir nueva fila al azar
        hijo[col_mutar] = fila_nueva        # Reemplazar el gen
        
    return hijo

---
### Funciones de Visualización y Algoritmos Completos
Para poder graficar el progreso, primero definimos la función que dibuja el tablero y las implementaciones completas de los algoritmos que devuelven el historial de estados.

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.animation as animation
from IPython.display import HTML

def graficar_tablero(estado, titulo, ax):
    """ Dibuja el tablero de ajedrez y coloca las reinas """
    tablero = np.zeros((8, 8))
    tablero[1::2, ::2] = 1
    tablero[::2, 1::2] = 1
    
    ax.imshow(tablero, cmap='gray_r', vmin=0, vmax=1)
    
    for col, fila in enumerate(estado):
        color_reina = 'red' if tablero[fila, col] == 0 else 'blue'
        ax.text(col, fila, '♕', fontsize=35, ha='center', va='center', color=color_reina)
        
    ax.set_title(titulo, fontweight='bold', fontsize=12)
    ax.set_xticks(range(8))
    ax.set_yticks(range(8))
    ax.set_xticklabels(['A', 'B', 'C', 'D', 'E', 'F', 'G', 'H'])
    ax.set_yticklabels([8, 7, 6, 5, 4, 3, 2, 1])

def ags_8reinas_completo(pop_size=150, max_generations=1000, mut_rate=0.2):
    pop = [np.random.randint(0, 8, 8).tolist() for _ in range(pop_size)]
    best_estado = None
    best_ataques = float('inf')
    historial_ataques = []
    historial_estados = []
    
    for gen in range(max_generations):
        ataques_pop = [calcular_ataques(ind) for ind in pop]
        min_ataques = min(ataques_pop)
        if min_ataques < best_ataques:
            best_ataques = min_ataques
            best_estado = pop[ataques_pop.index(min_ataques)].copy()
            
        historial_ataques.append(best_ataques)
        historial_estados.append(best_estado.copy())
        
        if best_ataques == 0: break
            
        new_pop = [best_estado.copy()]
        while len(new_pop) < pop_size:
            p1 = seleccion_torneo(pop, ataques_pop)
            p2 = seleccion_torneo(pop, ataques_pop)
            hijo = combinacion_un_punto(p1, p2)
            hijo = mutacion(hijo, mut_rate)
            new_pop.append(hijo)
        pop = new_pop
        
    return best_estado, best_ataques, historial_ataques, historial_estados

def tabu_8reinas_completo(max_iter=200, tabu_tenure=12):
    estado_actual = np.random.randint(0, 8, 8).tolist()
    mejor_estado = estado_actual.copy()
    mejor_ataques = calcular_ataques(mejor_estado)
    lista_tabu = [] 
    historial_ataques = []
    historial_estados = []
    
    for i in range(max_iter):
        historial_ataques.append(mejor_ataques)
        historial_estados.append(mejor_estado.copy())
        if mejor_ataques == 0: break
            
        vecinos = []
        for col in range(8):
            for fila in range(8):
                if estado_actual[col] != fila:
                    vecino = estado_actual.copy()
                    vecino[col] = fila
                    vecinos.append((vecino, col, fila))
                    
        vecinos_evaluados = [(vec, c, f, calcular_ataques(vec)) for vec, c, f in vecinos]
        vecinos_evaluados.sort(key=lambda x: x[3])
        
        mov_elegido = None
        for vec, c, f, atq in vecinos_evaluados:
            if atq < mejor_ataques or (c, f) not in lista_tabu:
                mov_elegido = (vec, (c, f), atq)
                break
        
        if mov_elegido:
            estado_actual, mov, atq_actual = mov_elegido
            if atq_actual < mejor_ataques:
                mejor_ataques = atq_actual
                mejor_estado = estado_actual.copy()
            lista_tabu.append(mov)
            if len(lista_tabu) > tabu_tenure: lista_tabu.pop(0)
            
    return mejor_estado, mejor_ataques, historial_ataques, historial_estados

---
### Animación de la Búsqueda (Paso a Paso)
**Nota sobre el error en Notebooks:** En entornos como Jupyter o Google Colab, la función `plt.show()` dentro de `FuncAnimation` a menudo no muestra nada o falla. La solución correcta es convertir la animación a HTML (JavaScript) para que el navegador pueda renderizarla.

In [ ]:
# Ejecutamos las búsquedas
sol_ags, atq_ags, hist_atq_ags, hist_est_ags = ags_8reinas_completo(max_generations=100)
sol_tabu, atq_tabu, hist_atq_tabu, hist_est_tabu = tabu_8reinas_completo(max_iter=100)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))
total_frames = max(len(hist_est_ags), len(hist_est_tabu))

def update(frame):
    ax1.clear()
    ax2.clear()
    
    # AGS
    idx_ags = min(frame, len(hist_est_ags) - 1)
    graficar_tablero(hist_est_ags[idx_ags], f'AGS - Gen {idx_ags}\nAmenazas: {hist_atq_ags[idx_ags]}', ax1)
    
    # TABU
    idx_tabu = min(frame, len(hist_est_tabu) - 1)
    graficar_tablero(hist_est_tabu[idx_tabu], f'TABU - Iter {idx_tabu}\nAmenazas: {hist_atq_tabu[idx_tabu]}', ax2)
    
    plt.tight_layout()

# Creamos la animación
ani = animation.FuncAnimation(fig, update, frames=total_frames, interval=200, repeat=False)

# Cerramos el plot estático para evitar que aparezca una imagen vacía antes de la animación
plt.close()

# Renderizamos la animación como HTML para que funcione en el Notebook
HTML(ani.to_jshtml())